<a href="https://colab.research.google.com/github/GwenONERA/VisualisationResultsEmotion/blob/main/reproductibilit%C3%A9_du_mod%C3%A8le_EMOTYC_online.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# étape 1 : renommer les noms des colonnes du fichier

In [1]:
import pandas as pd
df = pd.read_excel('/content/merged_text1.xlsx')
df.columns

Index(['Unnamed: 0', 'id_texte', 'document', 'sent_id', 'sentence',
       'score_de_polarite', 'score_de_subjectitvite', 'neutre', 'admiration',
       'amour', 'apaisement', 'audace', 'colere', 'comportement',
       'culpabilite', 'degout', 'deplaisir', 'desir', 'embarras', 'empathie',
       'fierte', 'impassibilite', 'inhumanite', 'jalousie', 'joie', 'mepris',
       'non_specifiee', 'orgueil', 'peur', 'ressentiment', 'surprise',
       'tristesse'],
      dtype='object')

In [ ]:
df.columns = [col.replace('nombre_avec_cat_', '') for col in df2.columns]

print("Nouveaux intitulés des colonnes pour df2 :")
display(df.columns)

In [ ]:
df.columns = [col.replace('nombre_avec_', '') for col in df2.columns]

print("Nouveaux intitulés des colonnes pour df2 :")
display(df.columns)

# étape 2 : reproductibilité des outputs

In [ ]:
import torch, numpy as np, pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("camembert-base")
model = (
    AutoModelForSequenceClassification
    .from_pretrained("TextToKids/CamemBERT-base-EmoTextToKids")
    .to(DEVICE)
    .eval()
)

df        = pd.read_excel("./processed_predictions.xlsx")
sentences = df["sentence"].astype(str).tolist()
EOS       = tokenizer.eos_token                            # </s>

# LABEL_COLS[j] = colonne XLSX correspondant à l'index j du modèle (id2label)
LABEL_COLS = [
    "emo", "emo_comportementale", "emo_designee", "emo_montree",
    "emo_suggeree", "emo_base", "emo_complexe",
    "admiration", "autre",
    "colere", "culpabilite", "degout", "embarras",
    "fierte", "jalousie", "joie", "peur", "surprise", "tristesse",
]
Y = df[LABEL_COLS].values.astype(int)                      # (101, 19)
N, L = Y.shape
print(f"Référence : {N} phrases × {L} labels = {N*L} cellules\n")

# ═══════════════════════════════════════════════════════════════
#  2. FORMATS D'ENTRÉE CANDIDATS
#     (on ne sait pas comment le web formate avant d'envoyer
#      au tokenizer → on teste plusieurs variantes)
# ═══════════════════════════════════════════════════════════════
TEMPLATES = {
    "raw":
        lambda s: s,
    "bca_eos_nospace":
        lambda s: f"before: {EOS}current: {s}{EOS}after: {EOS}",
    "bca_eos_space":
        lambda s: f"before: {EOS} current: {s} after: {EOS}",
    "bca_empty":
        lambda s: f"before:  current: {s} after: ",
    "bca_none_str":
        lambda s: f"before: None current: {s} after: None",
}

# ═══════════════════════════════════════════════════════════════
#  3. INFÉRENCE  –  sigmoid (multi-label), dropout désactivé
# ═══════════════════════════════════════════════════════════════
@torch.no_grad()
def infer_sigmoid(template_fn):
    """Renvoie la matrice (N, L) des probabilités sigmoïdes."""
    P = np.empty((N, L), dtype=np.float64)
    for i, s in enumerate(sentences):
        enc = tokenizer(
            template_fn(s),
            return_tensors="pt",
            truncation=True,
            max_length=512,
        ).to(DEVICE)
        logits = model(**enc).logits.squeeze(0)          # (19,)
        P[i] = torch.sigmoid(logits).cpu().double().numpy()
    return P

# ═══════════════════════════════════════════════════════════════
#  4. OPTIMISATION DES SEUILS (un par label)
#     Pour chaque label j on parcourt tous les « midpoints »
#     entre valeurs consécutives de σ(logit_j) et on garde
#     celui qui minimise les erreurs binaires.
# ═══════════════════════════════════════════════════════════════
def sweep_thresholds(P, Y):
    thresholds = np.empty(L)
    total_err  = 0
    for j in range(L):
        pj, yj = P[:, j], Y[:, j]
        vals = np.sort(np.unique(pj))
        # seuils candidats : entre chaque paire de probas + bornes
        cuts = np.concatenate([
            [vals[0] - 1e-9],
            (vals[:-1] + vals[1:]) / 2.0,
            [vals[-1] + 1e-9],
        ])
        errs = np.array([
            int(((pj >= t).astype(int) != yj).sum()) for t in cuts
        ])
        idx_best = int(errs.argmin())
        thresholds[j] = cuts[idx_best]
        total_err += errs[idx_best]
    return thresholds, total_err

# ═══════════════════════════════════════════════════════════════
#  5. RECHERCHE DU MEILLEUR FORMAT
# ═══════════════════════════════════════════════════════════════
winner = dict(err=N * L + 1)

for name, fn in TEMPLATES.items():
    P   = infer_sigmoid(fn)
    thr, err_opt = sweep_thresholds(P, Y)
    err_05 = int(((P >= 0.5).astype(int) != Y).sum())

    star = " ★ PERFECT" if err_opt == 0 else ""
    print(f"  {name:20s}  err@0.5={err_05:4d}   "
          f"err@opt={err_opt:4d}  /{N*L}{star}")

    if err_opt < winner["err"]:
        winner = dict(name=name, fn=fn, P=P, thr=thr, err=err_opt)
    if err_opt == 0:                       # inutile de continuer
        break

# ═══════════════════════════════════════════════════════════════
#  6. RAPPORT DÉTAILLÉ
# ═══════════════════════════════════════════════════════════════
print(f"\n{'═'*62}")
print(f"  Meilleur template  : {winner['name']}")
print(f"  Erreurs restantes  : {winner['err']} / {N*L}")
print(f"{'═'*62}\n")

P, thr = winner["P"], winner["thr"]

print("Seuils optimaux par label :")
for j in range(L):
    preds  = (P[:, j] >= thr[j]).astype(int)
    ne     = int((preds != Y[:, j]).sum())
    status = "✓" if ne == 0 else f"✗ {ne} erreur(s)"
    print(f"  {LABEL_COLS[j]:<25s}  seuil = {thr[j]:.8f}   {status}")

Y_hat  = (P >= thr).astype(int)
row_ok = int((Y_hat == Y).all(axis=1).sum())
print(f"\nLignes parfaitement reproduites : {row_ok} / {N}")
print(f"Précision cellule par cellule  : "
      f"{(Y_hat == Y).sum()} / {N*L}  "
      f"({(Y_hat == Y).mean():.4%})")

# Détail des éventuelles erreurs résiduelles
bad = np.where(~(Y_hat == Y).all(axis=1))[0]
if bad.size:
    print(f"\nLignes en erreur ({bad.size}) :")
    for i in bad:
        for j in np.where(Y_hat[i] != Y[i])[0]:
            print(f"  row {i:3d}  {LABEL_COLS[j]:<22s}  "
                  f"prédit={Y_hat[i,j]}  vrai={Y[i,j]}  "
                  f"σ={P[i,j]:.8f}")
else:
    print("\n✅  Aucune erreur : les outputs web sont parfaitement reproduits !")

# ═══════════════════════════════════════════════════════════════
#  7. FONCTION RÉUTILISABLE  (à brancher dans votre pipeline)
# ═══════════════════════════════════════════════════════════════
FINAL_TEMPLATE   = winner["fn"]
FINAL_THRESHOLDS = winner["thr"]          # np.array de shape (19,)

@torch.no_grad()
def predict_binary(sentence: str) -> dict[str, int]:
    """
    Reproduit exactement la sortie binaire du modèle web
    pour une phrase isolée.
    """
    text = FINAL_TEMPLATE(sentence)
    enc  = tokenizer(text, return_tensors="pt",
                     truncation=True, max_length=512).to(DEVICE)
    logits = model(**enc).logits.squeeze(0)
    probs  = torch.sigmoid(logits).cpu().numpy()
    binary = (probs >= FINAL_THRESHOLDS).astype(int)
    return {LABEL_COLS[j]: int(binary[j]) for j in range(L)}


# ── Validation finale sur toutes les lignes ──
print("\n\nValidation complète :")
all_ok = True
for i in range(N):
    pred = predict_binary(sentences[i])
    true = {LABEL_COLS[j]: int(Y[i, j]) for j in range(L)}
    if pred != true:
        all_ok = False
        print(f"  ✗ row {i}: {sentences[i][:55]}…")
if all_ok:
    print("  ✓ 101/101 lignes identiques aux outputs web.")

# ── Export des seuils pour réutilisation ──
thresholds_dict = {LABEL_COLS[j]: float(FINAL_THRESHOLDS[j]) for j in range(L)}
print(f"\n# Seuils à copier-coller :\nTHRESHOLDS = {thresholds_dict}")